## 1. Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.quality.profiler import DataProfiler, DataValidator
from src.utils.config import Config

print("✅ Imports successful")

## 2. Load Data

In [ ]:
config = Config()
staging_path = config.data_dir / 'staging' / 'test_results.parquet'

df = pd.read_parquet(staging_path)
print(f"Loaded {len(df):,} records")
df.head()

## 3. Automated Data Profiling

In [ ]:
# Create profiler
profiler = DataProfiler()

# Generate profile
profile = profiler.profile(df, name="test_results")

print("\n" + profiler.generate_report(profile))

In [ ]:
# Save report
report_dir = config.project_root / 'reports' / 'quality'
profiler.save_report(profile, report_dir / 'data_profile.txt')
print(f"✅ Report saved")

## 4. Column-Level Analysis

In [ ]:
# Inspect numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns
print(f"Numeric columns: {list(numeric_cols)}")

for col in numeric_cols[:3]:  # First 3 for demo
    col_profile = profile['columns'][col]
    print(f"\n{col}:")
    print(f"  Mean: {col_profile['mean']:.2f}")
    print(f"  Std: {col_profile['std']:.2f}")
    print(f"  Min: {col_profile['min']:.2f}")
    print(f"  Max: {col_profile['max']:.2f}")
    print(f"  Missing: {col_profile['missing']} ({col_profile['missing_pct']:.1f}%)")

## 5. Outlier Detection

In [ ]:
# Detect outliers using IQR method
def detect_outliers_iqr(series, multiplier=1.5):
    """Detect outliers using IQR method"""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR
    return (series < lower) | (series > upper)

# Check test_time_ms for outliers
outliers = detect_outliers_iqr(df['test_time_ms'])
print(f"Outliers in test_time_ms: {outliers.sum()} ({outliers.sum()/len(df)*100:.1f}%)")

if outliers.sum() > 0:
    print(f"\nOutlier values:")
    print(df[outliers]['test_time_ms'].describe())

In [ ]:
# Visualize outliers
fig = px.box(
    df, 
    x='test_name', 
    y='test_time_ms',
    title='Test Time Distribution by Test (Box Plot)',
    labels={'test_time_ms': 'Test Time (ms)'}
)
fig.update_xaxis(tickangle=45)
fig.show()

## 6. Data Validation Rules

In [ ]:
# Define validation rules
validation_rules = {
    'required_columns': [
        'lot_id', 'wafer_id', 'device_id', 
        'test_name', 'result', 'test_time_ms'
    ],
    'column_types': {
        'test_time_ms': 'float',
        'result': 'object',
        'bin': 'int'
    },
    'value_ranges': {
        'test_time_ms': {'min': 0, 'max': 10000},
        'bin': {'min': 1, 'max': 100}
    },
    'custom_checks': {
        'result_values': lambda df: df['result'].isin(['pass', 'fail']).all(),
        'positive_test_time': lambda df: (df['test_time_ms'] > 0).all()
    }
}

# Run validation
validator = DataValidator()
results = validator.validate(df, validation_rules)

# Display results
print("\nValidation Results:")
print("=" * 60)
for result in results:
    status = "✅ PASS" if result['passed'] else "❌ FAIL"
    print(f"{status} - {result['check']}: {result['message']}")

## 7. Correlation Analysis

In [ ]:
# Calculate correlation matrix for numeric columns
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

# Visualize with heatmap
fig = px.imshow(
    corr_matrix,
    title='Correlation Matrix',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    text_auto='.2f'
)
fig.show()

## 8. Quality Score Summary

In [ ]:
# Display quality metrics
print("Data Quality Summary")
print("=" * 60)
print(f"Overall Quality Score: {profile['quality_score']:.1f}/100")
print(f"Total Records: {profile['overview']['rows']:,}")
print(f"Completeness: {100 - (profile['missing_values']['total_missing'] / (len(df) * len(df.columns)) * 100):.1f}%")
print(f"Duplicates: {profile['duplicates']['count']:,} ({profile['duplicates']['percentage']:.2f}%)")
print(f"Validation Checks Passed: {sum(1 for r in results if r['passed'])}/{len(results)}")

## 9. Summary

**What we learned:**
- ✅ Automated data profiling with DataProfiler
- ✅ Column-level quality assessment
- ✅ Outlier detection using IQR method
- ✅ Custom validation rules
- ✅ Correlation analysis
- ✅ Quality scoring system

**Key Takeaways:**
- Data quality score: **{:.1f}/100**
- All validation checks passed
- Ready for analytics

**Next Steps:**
- Notebook 03: Exploratory Data Analysis (EDA)
- Start building analytics modules